In [1]:
import sys
import subprocess
import os
import json
import time
import datetime
import platform
from dataclasses import dataclass, asdict
from typing import Dict, Any, Optional

def install_requirements():
    required_packages = [
        'pandas',
        'numpy',
        'pyarrow',
        'fastparquet'
    ]

    for package in required_packages:
        try:
            __import__(package)
        except ImportError:
            print(f"Installing {package}")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", package]
            )

    print("====")
    print("All dependencies installed")
    print("====")


install_requirements()

import numpy as np
import pandas as pd

====
All dependencies installed
====


In [2]:
@dataclass
class FinancialConfig:
    profit_margin: float = 0.10
    chargeback_fee: float = 15.0
    false_positive_cost: float = 5.0
    review_cost: float = 2.50
    review_fraud_capture_rate: float = 0.90
    review_legit_approval_rate: float = 0.90

    def __post_init__(self):
        if not 0.0 < self.profit_margin < 1.0:
            raise ValueError(
                "Profit margin must be between 0 and 1."
            )

        if self.chargeback_fee < 0:
            raise ValueError(
                "Chargeback fee cannot be negative."
            )

        if self.false_positive_cost < 0:
            raise ValueError(
                "False positive cost cannot be negative."
            )

        if self.review_cost < 0:
            raise ValueError(
                "Review cost cannot be negative."
            )

        if not 0.0 <= self.review_fraud_capture_rate <= 1.0:
            raise ValueError(
                "Review fraud capture rate must be between 0 and 1."
            )

        if not 0.0 <= self.review_legit_approval_rate <= 1.0:
            raise ValueError(
                "Review legitimate approval rate must be between 0 and 1."
            )


FIN_CONFIG = FinancialConfig()

CALIB_DATA_PATH = "calib_m5.parquet"
WINNING_PREDS_PATH = "m5_winning_predictions.csv"

COST_OPTIMIZER_REPORT = "tri_state_cost_report.json"
COST_LANDSCAPE_DATA = "cost_landscape_matrix.csv"
THRESHOLD_SENSITIVITY_DATA = "threshold_sensitivity.csv"

In [4]:
def validate_and_load_financial_state(
    calib_path: str,
    preds_path: str
) -> pd.DataFrame:

    print("Initializing Financial Optimization Engine")
    
    if not os.path.exists(calib_path):
        raise FileNotFoundError(
            f"Calibration data missing: {calib_path}"
        )

    if not os.path.exists(preds_path):
        raise FileNotFoundError(
            f"Prediction file missing: {preds_path}"
        )

    print(
        f"Loading calibration data: {calib_path}"
    )

    calib_df = pd.read_parquet(
        calib_path
    )

    print(
        f"Loading winning predictions: {preds_path}"
    )

    preds_df = pd.read_csv(
        preds_path
    )

    required_prediction_columns = [
        'actual_fraud',
        'fraud_probability'
    ]

    for col in required_prediction_columns:
        if col not in preds_df.columns:
            raise KeyError(
                f"Prediction file missing required column: {col}"
            )

    if len(calib_df) != len(preds_df):
        raise ValueError(
            f"Row count mismatch: "
            f"M5={len(calib_df):,}, "
            f"Predictions={len(preds_df):,}"
        )

    if 'TransactionAmt' not in calib_df.columns:
        raise KeyError(
            "TransactionAmt missing from calibration data."
        )

    labels_from_data = (
        calib_df['isFraud']
        .astype(np.int8)
        .to_numpy()
    )

    labels_from_predictions = (
        preds_df['actual_fraud']
        .astype(np.int8)
        .to_numpy()
    )

    if not np.array_equal(
        labels_from_data,
        labels_from_predictions
    ):
        raise ValueError(
            "Target alignment failure: "
            "prediction labels do not match calibration labels."
        )

    probabilities = (
        pd.to_numeric(
            preds_df['fraud_probability'],
            errors='coerce'
        )
        .to_numpy(
            dtype=np.float32
        )
    )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            "Prediction probabilities contain NaN or infinite values."
        )

    if (
        (probabilities < 0).any()
        or
        (probabilities > 1).any()
    ):
        raise ValueError(
            "Prediction probabilities must lie in [0, 1]."
        )

    if 'TransactionID' in calib_df.columns:
        transaction_ids = (
            calib_df['TransactionID']
            .to_numpy()
        )
    else:
        transaction_ids = np.arange(
            len(calib_df)
        )

    finance_df = pd.DataFrame({
        'transaction_id': transaction_ids,
        'amount': calib_df['TransactionAmt']
            .astype(np.float32),
        'actual_fraud': labels_from_data,
        'fraud_prob': probabilities
    })

    if (finance_df['amount'] < 0).any():
        raise ValueError(
            "Transaction amounts cannot be negative."
        )

    total_value = (
        finance_df['amount']
        .sum()
    )

    fraud_count = int(
        finance_df['actual_fraud']
        .sum()
    )

    fraud_rate = (
        fraud_count / len(finance_df)
        if len(finance_df) > 0
        else 0.0
    )

    print("FINANCIAL STATE VERIFIED")
    print(
        f"Transactions:              {len(finance_df):,}"
    )
    print(
        f"Portfolio Value:           ${total_value:,.2f}"
    )
    print(
        f"Fraudulent Transactions:   {fraud_count:,}"
    )
    print(
        f"Fraud Rate:                {fraud_rate * 100:.2f}%"
    )

    return finance_df


portfolio_state_df = (
    validate_and_load_financial_state(
        CALIB_DATA_PATH,
        WINNING_PREDS_PATH
    )
)

Initializing Financial Optimization Engine
Loading calibration data: calib_m5.parquet
Loading winning predictions: m5_winning_predictions.csv
FINANCIAL STATE VERIFIED
Transactions:              8,858
Portfolio Value:           $1,159,238.50
Fraudulent Transactions:   281
Fraud Rate:                3.17%


In [5]:
VEC_AMOUNTS = np.ascontiguousarray(
    portfolio_state_df['amount']
    .to_numpy(dtype=np.float32)
)

VEC_LABELS = np.ascontiguousarray(
    portfolio_state_df['actual_fraud']
    .to_numpy(dtype=np.int8)
)

VEC_PROBS = np.ascontiguousarray(
    portfolio_state_df['fraud_prob']
    .to_numpy(dtype=np.float32)
)

print("Financial vectors loaded into contiguous NumPy arrays")


Financial vectors loaded into contiguous NumPy arrays


In [6]:
def calculate_tri_state_cost_vectorized(
    amounts: np.ndarray,
    labels: np.ndarray,
    probs: np.ndarray,
    t_low: float,
    t_high: float,
    config: FinancialConfig
) -> Dict[str, Any]:

    if not 0.0 <= t_low <= 1.0:
        raise ValueError(
            "t_low must be between 0 and 1."
        )

    if not 0.0 <= t_high <= 1.0:
        raise ValueError(
            "t_high must be between 0 and 1."
        )

    if t_low > t_high:
        raise ValueError(
            "t_low cannot exceed t_high."
        )

    approve_mask = probs < t_low

    review_mask = (
        (probs >= t_low)
        &
        (probs <= t_high)
    )

    reject_mask = probs > t_high

    fraud_mask = labels == 1
    legit_mask = labels == 0

    approve_fraud = (
        approve_mask & fraud_mask
    )

    reject_legit = (
        reject_mask & legit_mask
    )

    review_fraud = (
        review_mask & fraud_mask
    )

    review_legit = (
        review_mask & legit_mask
    )

    approve_fraud_count = int(
        approve_fraud.sum()
    )

    reject_legit_count = int(
        reject_legit.sum()
    )

    review_fraud_count = int(
        review_fraud.sum()
    )

    review_legit_count = int(
        review_legit.sum()
    )

    review_fraud_caught = int(
        round(
            review_fraud_count
            * config.review_fraud_capture_rate
        )
    )

    review_fraud_missed = (
        review_fraud_count
        - review_fraud_caught
    )

    review_legit_approved = int(
        round(
            review_legit_count
            * config.review_legit_approval_rate
        )
    )

    review_legit_rejected = (
        review_legit_count
        - review_legit_approved
    )

    approve_fraud_amount = float(
        amounts[approve_fraud].sum()
    )

    review_fraud_missed_amount = 0.0

    if review_fraud_count > 0:
        review_fraud_missed_amount = (
            amounts[review_fraud].sum()
            * (
                review_fraud_missed
                / review_fraud_count
            )
        )

    reject_legit_amount = float(
        amounts[reject_legit].sum()
    )

    review_legit_rejected_amount = 0.0

    if review_legit_count > 0:
        review_legit_rejected_amount = (
            amounts[review_legit].sum()
            * (
                review_legit_rejected
                / review_legit_count
            )
        )

    false_negative_count = (
        approve_fraud_count
        + review_fraud_missed
    )

    false_positive_count = (
        reject_legit_count
        + review_legit_rejected
    )

    false_negative_cost = (
        approve_fraud_amount
        +
        (
            approve_fraud_count
            * config.chargeback_fee
        )
        +
        review_fraud_missed_amount
        +
        (
            review_fraud_missed
            * config.chargeback_fee
        )
    )

    false_positive_cost = (
        reject_legit_amount
        * config.profit_margin
        +
        reject_legit_count
        * config.false_positive_cost
        +
        review_legit_rejected_amount
        * config.profit_margin
        +
        review_legit_rejected
        * config.false_positive_cost
    )

    operational_review_cost = (
        int(review_mask.sum())
        * config.review_cost
    )

    total_cost = (
        false_negative_cost
        +
        false_positive_cost
        +
        operational_review_cost
    )

    return {
        't_low': float(t_low),
        't_high': float(t_high),
        'total_cost': float(total_cost),
        'fn_cost': float(false_negative_cost),
        'fp_cost': float(false_positive_cost),
        'review_cost': float(
            operational_review_cost
        ),
        'false_negative_count': int(
            false_negative_count
        ),
        'false_positive_count': int(
            false_positive_count
        ),
        'auto_approved': int(
            approve_mask.sum()
        ),
        'sent_to_review': int(
            review_mask.sum()
        ),
        'auto_rejected': int(
            reject_mask.sum()
        ),
        'fraud_caught_in_review': int(
            review_fraud_caught
        ),
        'fraud_missed_in_review': int(
            review_fraud_missed
        )
    }

In [8]:
def calculate_baseline_strategies(
    amounts: np.ndarray,
    labels: np.ndarray,
    config: FinancialConfig
) -> Dict[str, float]:

    fraud_mask = labels == 1
    legit_mask = labels == 0

    fraud_amount = float(
        amounts[fraud_mask].sum()
    )

    legit_amount = float(
        amounts[legit_mask].sum()
    )

    fraud_count = int(
        fraud_mask.sum()
    )

    legit_count = int(
        legit_mask.sum()
    )

    approve_all_cost = (
        fraud_amount
        +
        fraud_count
        * config.chargeback_fee
    )

    block_all_cost = (
        legit_amount
        * config.profit_margin
        +
        legit_count
        * config.false_positive_cost
    )

    return {
        'approve_all': float(
            approve_all_cost
        ),
        'block_all': float(
            block_all_cost
        )
    }


baseline_losses = calculate_baseline_strategies(
    VEC_AMOUNTS,
    VEC_LABELS,
    FIN_CONFIG
)

print("BASELINE STRATEGIES")
print(
    f"Approve Everything Loss:   "
    f"${baseline_losses['approve_all']:,.2f}"
)
print(
    f"Block Everything Loss:     "
    f"${baseline_losses['block_all']:,.2f}"
)
print("====")

BASELINE STRATEGIES
Approve Everything Loss:   $54,830.79
Block Everything Loss:     $153,747.28
====


In [10]:
def optimize_financial_thresholds(
    amounts: np.ndarray,
    labels: np.ndarray,
    probs: np.ndarray,
    config: FinancialConfig
) -> Dict[str, Any]:

    print("Executing Vectorized Threshold Optimization")
    
    start_time = time.time()

    thresholds = np.linspace(
        0.01,
        0.99,
        99,
        dtype=np.float32
    )

    landscape_data = []

    best_metrics = None
    best_cost = float('inf')

    iterations = 0

    for t_low in thresholds:

        valid_highs = (
            thresholds[
                thresholds >= t_low
            ]
        )

        for t_high in valid_highs:

            metrics = calculate_tri_state_cost_vectorized(
                amounts=amounts,
                labels=labels,
                probs=probs,
                t_low=float(t_low),
                t_high=float(t_high),
                config=config
            )

            landscape_data.append({
                't_low': metrics['t_low'],
                't_high': metrics['t_high'],
                'total_cost': metrics['total_cost']
            })

            iterations += 1

            if metrics['total_cost'] < best_cost:
                best_cost = metrics['total_cost']
                best_metrics = metrics

    elapsed = time.time() - start_time

    landscape_df = pd.DataFrame(
        landscape_data
    )

    landscape_df.to_csv(
        COST_LANDSCAPE_DATA,
        index=False
    )

    approve_all_loss = (
        baseline_losses['approve_all']
    )

    block_all_loss = (
        baseline_losses['block_all']
    )

    net_savings_vs_approve_all = (
        approve_all_loss
        -
        best_metrics['total_cost']
    )

    net_savings_vs_best_static = (
        min(
            approve_all_loss,
            block_all_loss
        )
        -
        best_metrics['total_cost']
    )

    loss_reduction_vs_approve_all = (
        (
            net_savings_vs_approve_all
            /
            approve_all_loss
        )
        * 100
        if approve_all_loss > 0
        else 0.0
    )

    loss_reduction_vs_best_static = (
        (
            net_savings_vs_best_static
            /
            min(
                approve_all_loss,
                block_all_loss
            )
        )
        * 100
        if min(
            approve_all_loss,
            block_all_loss
        ) > 0
        else 0.0
    )

    print("EXECUTIVE FINANCIAL REPORT")
    print(
        f"Threshold Combinations:    {iterations:,}"
    )
    print(
        f"Optimization Time:         {elapsed:.4f} seconds"
    )
    print(
        f"Approve Everything Loss:   "
        f"${approve_all_loss:,.2f}"
    )
    print(
        f"Block Everything Loss:     "
        f"${block_all_loss:,.2f}"
    )
    print(
        f"Optimized Tri-State Loss:   "
        f"${best_metrics['total_cost']:,.2f}"
    )
    print(
        f"Savings vs Approve All:    "
        f"${net_savings_vs_approve_all:,.2f}"
    )
    print(
        f"Loss Reduction vs Approve: "
        f"{loss_reduction_vs_approve_all:.2f}%"
    )
    print(
        f"Savings vs Best Static:    "
        f"${net_savings_vs_best_static:,.2f}"
    )
    print(
        f"Loss Reduction vs Static:  "
        f"{loss_reduction_vs_best_static:.2f}%"
    )
    print("OPTIMAL ROUTING THRESHOLDS")
    print(
        f"T_low:                     "
        f"{best_metrics['t_low']:.3f}"
    )
    print(
        f"T_high:                    "
        f"{best_metrics['t_high']:.3f}"
    )
    print("PORTFOLIO ROUTING VOLUME")
    print(
        f"Auto-Approved:              "
        f"{best_metrics['auto_approved']:,}"
    )
    print(
        f"Sent to Review:             "
        f"{best_metrics['sent_to_review']:,}"
    )
    print(
        f"Auto-Rejected:              "
        f"{best_metrics['auto_rejected']:,}"
    )
    print("COST BREAKDOWN")
    print(
        f"False Negative Cost:       "
        f"${best_metrics['fn_cost']:,.2f}"
    )
    print(
        f"False Positive Cost:       "
        f"${best_metrics['fp_cost']:,.2f}"
    )
    print(
        f"Review Cost:               "
        f"${best_metrics['review_cost']:,.2f}"
    )

    result = dict(best_metrics)

    result.update({
        'baseline_loss_approve_all': float(
            approve_all_loss
        ),
        'baseline_loss_block_all': float(
            block_all_loss
        ),
        'net_savings_vs_approve_all': float(
            net_savings_vs_approve_all
        ),
        'net_savings_vs_best_static': float(
            net_savings_vs_best_static
        ),
        'loss_reduction_vs_approve_all_pct': float(
            loss_reduction_vs_approve_all
        ),
        'loss_reduction_vs_best_static_pct': float(
            loss_reduction_vs_best_static
        ),
        'threshold_combinations_evaluated': int(
            iterations
        ),
        'optimization_time_seconds': float(
            elapsed
        )
    })

    return result


optimal_financial_strategy = optimize_financial_thresholds(
    amounts=VEC_AMOUNTS,
    labels=VEC_LABELS,
    probs=VEC_PROBS,
    config=FIN_CONFIG
)

Executing Vectorized Threshold Optimization
EXECUTIVE FINANCIAL REPORT
Threshold Combinations:    4,950
Optimization Time:         0.4978 seconds
Approve Everything Loss:   $54,830.79
Block Everything Loss:     $153,747.28
Optimized Tri-State Loss:   $22,672.22
Savings vs Approve All:    $32,158.57
Loss Reduction vs Approve: 58.65%
Savings vs Best Static:    $32,158.57
Loss Reduction vs Static:  58.65%
OPTIMAL ROUTING THRESHOLDS
T_low:                     0.210
T_high:                    0.520
PORTFOLIO ROUTING VOLUME
Auto-Approved:              6,421
Sent to Review:             2,310
Auto-Rejected:              127
COST BREAKDOWN
False Negative Cost:       $11,990.82
False Positive Cost:       $4,906.40
Review Cost:               $5,775.00


In [12]:
def run_threshold_sensitivity(
    amounts: np.ndarray,
    labels: np.ndarray,
    probs: np.ndarray,
    config: FinancialConfig,
    best_metrics: Dict[str, Any],
    radius: int = 2
):

    best_low_idx = int(
        round(
            best_metrics['t_low']
            * 100
        )
    )

    best_high_idx = int(
        round(
            best_metrics['t_high']
            * 100
        )
    )

    rows = []

    for low_idx in range(
        max(1, best_low_idx - radius),
        min(99, best_low_idx + radius) + 1
    ):

        for high_idx in range(
            max(low_idx, best_high_idx - radius),
            min(99, best_high_idx + radius) + 1
        ):

            t_low = low_idx / 100.0
            t_high = high_idx / 100.0

            metrics = calculate_tri_state_cost_vectorized(
                amounts,
                labels,
                probs,
                t_low,
                t_high,
                config
            )

            rows.append({
                't_low': t_low,
                't_high': t_high,
                'total_cost': metrics['total_cost'],
                'cost_delta_from_optimum': (
                    metrics['total_cost']
                    -
                    best_metrics['total_cost']
                )
            })

    sensitivity_df = pd.DataFrame(
        rows
    ).sort_values(
        'total_cost'
    )

    sensitivity_df.to_csv(
        THRESHOLD_SENSITIVITY_DATA,
        index=False
    )

    print("THRESHOLD SENSITIVITY")
    print(
        sensitivity_df.head(10)
        .to_string(index=False)
    )
    return sensitivity_df


sensitivity_df = run_threshold_sensitivity(
    VEC_AMOUNTS,
    VEC_LABELS,
    VEC_PROBS,
    FIN_CONFIG,
    optimal_financial_strategy
)

THRESHOLD SENSITIVITY
 t_low  t_high   total_cost  cost_delta_from_optimum
  0.21    0.52 22672.220703                 0.000000
  0.21    0.53 22675.519531                 3.298828
  0.21    0.50 22777.742188               105.521484
  0.21    0.51 22846.033203               173.812500
  0.20    0.52 23114.673828               442.453125
  0.20    0.51 23287.855469               615.634766
  0.20    0.53 23319.968750               647.748047
  0.21    0.54 23353.410156               681.189453
  0.20    0.50 23418.423828               746.203125
  0.19    0.52 23524.171875               851.951172


In [14]:
def summarize_policy_stability(
    sensitivity_df: pd.DataFrame,
    optimal_cost: float
) -> Dict[str, float]:

    best_cost = sensitivity_df[
        'total_cost'
    ].min()

    cost_1pct = (
        best_cost * 1.01
    )

    cost_5pct = (
        best_cost * 1.05
    )

    within_1pct = int(
        (
            sensitivity_df['total_cost']
            <= cost_1pct
        ).sum()
    )

    within_5pct = int(
        (
            sensitivity_df['total_cost']
            <= cost_5pct
        ).sum()
    )

    summary = {
        'sensitivity_points_within_1pct': within_1pct,
        'sensitivity_points_within_5pct': within_5pct,
        'sensitivity_best_cost': float(best_cost),
        'optimal_cost': float(optimal_cost)
    }

    print("POLICY STABILITY SUMMARY")
    print(
        f"Points within 1% of optimum: "
        f"{within_1pct}"
    )
    print(
        f"Points within 5% of optimum: "
        f"{within_5pct}"
    )

    return summary


policy_stability = summarize_policy_stability(
    sensitivity_df,
    optimal_financial_strategy['total_cost']
)

POLICY STABILITY SUMMARY
Points within 1% of optimum: 4
Points within 5% of optimum: 14


In [16]:
def export_financial_policy(
    metrics_dict: Dict[str, Any],
    config: FinancialConfig,
    report_path: str,
    source_predictions: str,
    sensitivity_summary: Dict[str, Any]
) -> None:

    print(
        f"Generating Financial Policy Artifact: "
        f"{report_path}"
    )

    policy_artifact = {
        "metadata": {
            "policy_version": "1.0",
            "generated_at_utc": (
                datetime.datetime.now(
                    datetime.timezone.utc
                ).isoformat()
            ),
            "environment": platform.system(),
            "source_prediction_artifact": source_predictions,
            "policy_type": "tri_state_transaction_risk"
        },

        "financial_assumptions": {
            "profit_margin_pct": config.profit_margin,
            "chargeback_fee_usd": config.chargeback_fee,
            "false_positive_cost_usd": (
                config.false_positive_cost
            ),
            "manual_review_cost_usd": (
                config.review_cost
            )
        },

        "review_assumptions": {
            "fraud_capture_rate": (
                config.review_fraud_capture_rate
            ),
            "legitimate_approval_rate": (
                config.review_legit_approval_rate
            )
        },

        "deployment_thresholds": {
            "auto_approve_max_risk": (
                metrics_dict['t_low']
            ),
            "manual_review_min_risk": (
                metrics_dict['t_low']
            ),
            "manual_review_max_risk": (
                metrics_dict['t_high']
            ),
            "auto_block_min_risk": (
                metrics_dict['t_high']
            )
        },

        "financial_performance": {
            "approve_all_loss_usd": (
                metrics_dict[
                    'baseline_loss_approve_all'
                ]
            ),
            "block_all_loss_usd": (
                metrics_dict[
                    'baseline_loss_block_all'
                ]
            ),
            "optimized_tri_state_loss_usd": (
                metrics_dict['total_cost']
            ),
            "net_savings_vs_approve_all_usd": (
                metrics_dict[
                    'net_savings_vs_approve_all'
                ]
            ),
            "net_savings_vs_best_static_usd": (
                metrics_dict[
                    'net_savings_vs_best_static'
                ]
            ),
            "loss_reduction_vs_approve_all_pct": (
                metrics_dict[
                    'loss_reduction_vs_approve_all_pct'
                ]
            ),
            "loss_reduction_vs_best_static_pct": (
                metrics_dict[
                    'loss_reduction_vs_best_static_pct'
                ]
            )
        },

        "cost_breakdown": {
            "false_negative_cost_usd": (
                metrics_dict['fn_cost']
            ),
            "false_positive_cost_usd": (
                metrics_dict['fp_cost']
            ),
            "review_operational_cost_usd": (
                metrics_dict['review_cost']
            )
        },

        "routing_volumes": {
            "auto_approved": (
                metrics_dict['auto_approved']
            ),
            "sent_to_review": (
                metrics_dict['sent_to_review']
            ),
            "auto_blocked": (
                metrics_dict['auto_rejected']
            ),
            "fraud_caught_in_review": (
                metrics_dict[
                    'fraud_caught_in_review'
                ]
            ),
            "fraud_missed_in_review": (
                metrics_dict[
                    'fraud_missed_in_review'
                ]
            )
        },

        "policy_stability": sensitivity_summary,

        "optimization": {
            "threshold_combinations_evaluated": (
                metrics_dict[
                    'threshold_combinations_evaluated'
                ]
            ),
            "optimization_time_seconds": (
                metrics_dict[
                    'optimization_time_seconds'
                ]
            )
        }
    }

    with open(
        report_path,
        'w',
        encoding='utf-8'
    ) as f:

        json.dump(
            policy_artifact,
            f,
            indent=4
        )

    print("Financial Policy Artifact Written Successfully")


export_financial_policy(
    metrics_dict=optimal_financial_strategy,
    config=FIN_CONFIG,
    report_path=COST_OPTIMIZER_REPORT,
    source_predictions=WINNING_PREDS_PATH,
    sensitivity_summary=policy_stability
)

Generating Financial Policy Artifact: tri_state_cost_report.json
Financial Policy Artifact Written Successfully


In [17]:


print(
    f"Auto-Approve Risk < "
    f"{optimal_financial_strategy['t_low']:.3f}"
)

print(
    f"Manual Review Risk: "
    f"{optimal_financial_strategy['t_low']:.3f}"
    f" to "
    f"{optimal_financial_strategy['t_high']:.3f}"
)

print(
    f"Auto-Block Risk > "
    f"{optimal_financial_strategy['t_high']:.3f}"
)

print(
    f"Optimized Loss: "
    f"${optimal_financial_strategy['total_cost']:,.2f}"
)

print(
    f"Loss Reduction vs Approve All: "
    f"{optimal_financial_strategy['loss_reduction_vs_approve_all_pct']:.2f}%"
)

print(
    f"Policy: "
    f"{COST_OPTIMIZER_REPORT}"
)


Auto-Approve Risk < 0.210
Manual Review Risk: 0.210 to 0.520
Auto-Block Risk > 0.520
Optimized Loss: $22,672.22
Loss Reduction vs Approve All: 58.65%
Policy: tri_state_cost_report.json
